# NYC Taxi & Limousine Commission (TLC) Diccionario de datos — NYC Yellow Taxi Trip Records

* NYC Taxi & Limousine Commission (TLC) Trip Record Data — es el portal oficial de datos abiertos de la ciudad de Nueva York. Los enlaces de descarga mes a mes; los archivos Parquet, pueden ser consumidos desde CloudFront sin necesidad de autenticación, pero se publican con 1-2 meses de rezago respecto al mes real. 
- Página oficial del dataset: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- Diccionario de datos oficial (PDF, fuente de este documento): https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
- Descarga de archivos mensuales (Parquet): `https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_AAAA-MM.parquet`

> Nota: algunas columnas se agregaron en distintos momentos (no existen en los archivos más antiguos): `improvement_surcharge` desde 2015, `congestion_surcharge` desde 2019, `airport_fee` en años recientes, y `cbd_congestion_fee` desde enero de 2025 (cargo por la zona de congestión de la MTA). Si vas a unir varios meses/años, valida el esquema de cada archivo antes de hacer un `UNION` o un `merge`.

## Campos

| Campo | Tipo | Descripción |
|---|---|---|
| `VendorID` | Numérico | Código de la empresa que proveyó el registro TPEP. Ver tabla de valores abajo. |
| `tpep_pickup_datetime` | DateTime | Fecha y hora en que se activó el taxímetro (inicio del viaje). |
| `tpep_dropoff_datetime` | DateTime | Fecha y hora en que se desactivó el taxímetro (fin del viaje). |
| `passenger_count` | Numérico | Número de pasajeros en el vehículo (valor ingresado por el conductor). |
| `trip_distance` | Numérico | Distancia recorrida en millas, reportada por el taxímetro. |
| `RatecodeID` | Numérico | Tarifa final vigente al terminar el viaje. Ver tabla de valores abajo. |
| `store_and_fwd_flag` | Carácter | Indica si el registro se guardó en la memoria del vehículo antes de transmitirse (por falta de conexión). `Y` = sí, `N` = no. |
| `PULocationID` | Numérico | Zona TLC en la que se activó el taxímetro (origen). Se une contra la tabla de zonas (`taxi_zone_lookup`). |
| `DOLocationID` | Numérico | Zona TLC en la que se desactivó el taxímetro (destino). |
| `payment_type` | Numérico | Cómo pagó el pasajero. Ver tabla de valores abajo. |
| `fare_amount` | Numérico | Tarifa calculada por el taxímetro en función de tiempo y distancia. |
| `extra` | Numérico | Recargos y extras varios (por ejemplo, tarifas de hora pico o nocturnas). |
| `mta_tax` | Numérico | Impuesto que se activa automáticamente según la tarifa medida en uso. |
| `tip_amount` | Numérico | Propina — se completa automáticamente para pagos con tarjeta; las propinas en efectivo no se incluyen. |
| `tolls_amount` | Numérico | Monto total de peajes pagados durante el viaje. |
| `improvement_surcharge` | Numérico | Recargo fijo aplicado al bajar la bandera; vigente desde 2015. |
| `total_amount` | Numérico | Monto total cobrado al pasajero. No incluye propinas en efectivo. |
| `congestion_surcharge` | Numérico | Recargo total por congestión del estado de Nueva York (NYS), vigente desde 2019. |
| `airport_fee` | Numérico | Cargo aplicado solo en recogidas en los aeropuertos LaGuardia y John F. Kennedy. |
| `cbd_congestion_fee` | Numérico | Cargo por viaje de la "Congestion Relief Zone" de la MTA, vigente desde el 5 de enero de 2025. |

## Valores codificados

### `VendorID`

| Valor | Significado |
|---|---|
| 1 | Creative Mobile Technologies, LLC |
| 2 | Curb Mobility, LLC |
| 6 | Myle Technologies Inc |
| 7 | Helix |

### `RatecodeID`

| Valor | Significado |
|---|---|
| 1 | Tarifa estándar |
| 2 | JFK |
| 3 | Newark |
| 4 | Nassau o Westchester |
| 5 | Tarifa negociada |
| 6 | Viaje grupal |
| 99 | Nulo / desconocido |

### `payment_type`

| Valor | Significado |
|---|---|
| 0 | Flex Fare |
| 1 | Tarjeta de crédito |
| 2 | Efectivo |
| 3 | Sin cargo |
| 4 | Disputa |
| 5 | Desconocido |
| 6 | Viaje anulado |

### `store_and_fwd_flag`

| Valor | Significado |
|---|---|
| `Y` | Store and forward trip — el registro se guardó en memoria antes de enviarse |
| `N` | No fue un store and forward trip |


# LIBRERIAS

In [12]:
import os
import requests
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from delta.tables import DeltaTable

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 14, Finished, Available, Finished, False)

# VARIABLES BASE

In [32]:
# ruta local del Lakehouse dentro del notebook
LAKEHOUSE_FILES = "/lakehouse/default/Files"
NAME_LH = "LH_BRONCE_ADVANCED"
NAME_SCHEMA = "NYCTLC"
TABLA_BRONZE = f"{NAME_LH}.{NAME_SCHEMA}.yellow_tripdata"

# parámetro: mes a ingestar
YYYY_MM = "2025-02"
URL_PUBLICA = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{YYYY_MM}.parquet"
PATH_SAVE = f"{LAKEHOUSE_FILES}/NYCTLC/{YYYY_MM}/yellow_tripdata_{YYYY_MM}.parquet"

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 34, Finished, Available, Finished, False)

# 1) Descargar el archivo público en el Lakehouse sin transformar

In [33]:
# Crear el directorio destino ANTES de escribir.
os.makedirs(os.path.dirname(PATH_SAVE), exist_ok=True)

# Descargar archivo
resp = requests.get(URL_PUBLICA, timeout=120)
resp.raise_for_status()
with open(PATH_SAVE, "wb") as f:
    f.write(resp.content)
print(f"Descargado: {URL_PUBLICA} -> {PATH_SAVE} ({len(resp.content) / (1024*1024):.1f} MB)")

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 35, Finished, Available, Finished, False)

Descargado: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-02.parquet -> /lakehouse/default/Files/NYCTLC/2025-02/yellow_tripdata_2025-02.parquet (57.5 MB)


# 2) Leer archivo crudo y agregar columnas de auditoria

In [34]:
LAKEHOUSE_FILES_SPARK = "Files/NYCTLC/"
FILE_SAVE = f"{YYYY_MM}/yellow_tripdata_{YYYY_MM}.parquet"

# Leer archivo parquet con spark
df = spark.read.parquet(LAKEHOUSE_FILES_SPARK +FILE_SAVE)
display(df.limit(10))

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0aa49eda-bf3b-4421-9930-e8f7abf3795f)

In [35]:
# agregar columnas de auditoria
df = df.withColumns({
    "pickup_year": F.year("tpep_pickup_datetime"),
    "pickup_month": F.month("tpep_pickup_datetime"),
    "url_file": F.lit(URL_PUBLICA),
    "ingested_at": F.current_timestamp()
})
display(df.limit(10))

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29c7a10d-c053-4e18-b0fe-14792e2a79fb)

# 3) Materializar TB en su respectivo schema y particionar TB por: 
* yyyy 
* MM
* replaceWhere reemplaza únicamente la partición correspondiente al periodo yyyy & MM, replaceWhere en Delta no es un simple filtro de partición — es una restricción que toda fila que escribes debe cumplir.

In [36]:
(
    df_raw.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"pickup_year = {YYYY_MM[:4]} AND pickup_month = {int(YYYY_MM[5:7])}")
    .partitionBy("pickup_year", "pickup_month")
    .saveAsTable(TABLA_BRONZE)
)

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 38, Finished, Available, Finished, False)

AnalysisException: [DELTA_REPLACE_WHERE_MISMATCH] Written data does not conform to partial table overwrite condition or constraint 'pickup_year = 2025 AND pickup_month = 2'.
[DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint EXPRESSION(('pickup_year = 2025)) (pickup_year = 2025) violated by row with values:
 - pickup_year : 2024

# 4) Validación mínima post-carga: filas cargadas y rango de fechas

# 

In [31]:
df_validate = spark.table(TABLA_BRONZE).filter(
    (F.col("pickup_year") == int(YYYY_MM[:4])) & (F.col("pickup_month") == int(YYYY_MM[5:7]))
).agg(
    F.count("*").alias("filas"),
    F.min("tpep_pickup_datetime").alias("min_fecha"),
    F.max("tpep_pickup_datetime").alias("max_fecha"),
)
df_validate.show(truncate=False)

StatementMeta(, 68c1cc93-d6cb-4eea-9e8a-44bbbc2e9ab2, 33, Finished, Available, Finished, False)

+-------+-------------------+-------------------+
|filas  |min_fecha          |max_fecha          |
+-------+-------------------+-------------------+
|3475204|2025-01-01 00:00:00|2025-01-31 23:59:59|
+-------+-------------------+-------------------+

